In [ ]:
# Load API keys - Direct from .env
import os
import sys

# API Keys (loaded directly - no dotenv needed)
API_KEYS = {
    'TWELVE_DATA_KEY': 'd19ebe6706614dd897e66aa416900fd3',
    'FINNHUB_KEY': 'd3qj8p9r01quv7kb49igd3qj8p9r01quv7kb49j0',
    'ALPHA_VANTAGE_KEY': 'gL_pHRAJ6SQK0AK2MD0rSuP653GW733l',
    'POLYGON_KEY': 'iRXh2jGpwhcJxGWfW4ZRVn2C4s_v4ghr',
    'FMP_KEY': '15zYYtksuJnQsTBODSNs3MrfEedOSd3i',
    'EODHD_KEY': '68f5419033db54.61168020',
}

print("✓ API Keys loaded directly")

# Show what we have
print("\nAPI Key Status:")
for key_name, key_value in API_KEYS.items():
    masked = key_value[:8] + '...' + key_value[-4:] if len(key_value) > 12 else '***'
    print(f"  ✓ {key_name}: {masked}")

print(f"\n✓ Total: {len(API_KEYS)}/6 API keys configured")
print("  Note: yfinance doesn't need a key (always works)")

## Step 1: Load API Keys from Environment
Attempts to load API keys from your config.py or environment variables.

In [ ]:
# Install all required packages
import sys
import subprocess

packages = [
    'yfinance',
    'requests',
    'finnhub-python',
    'pandas',
    'numpy',
    'python-dotenv'
]

print("Installing required packages...")
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"  ✓ {package} already installed")
    except ImportError:
        print(f"  Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        print(f"  ✓ {package} installed")

print("\n✓ All packages ready")

## Step 0: Install Required Packages
Run this first if you get import errors. Installs all packages needed for testing.

# 01 - API & Data Source Validation
**Mission:** Test ALL 7 free APIs with YOUR watchlist. Prove what works, reject what doesn't.

**Acceptance Criteria:**
- Latency: <5s per ticker
- Success Rate: >95% over 10 attempts
- Data Quality: No gaps in OHLCV for last 90 days
- Rate Limits: Document actual limits vs advertised

**Your Watchlist (9 Current Holdings):**
- IONQ, ASTS, APLD, HOOD, UBER, LYFT, LUNR, XBIO, KDK

**APIs Under Test:**
1. yfinance (Free, unlimited)
2. Twelve Data (800/day free)
3. Finnhub (60/min free)
4. Polygon (free tier)
5. Alpha Vantage (500/day free)
6. FMP (250/day free)
7. EODHD (20/day free)

**Test Duration:** ~15-30 minutes (all APIs, all tickers)

In [ ]:
# Setup & Imports
import sys
import time
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Your current holdings to test with
WATCHLIST = ['IONQ', 'ASTS', 'APLD', 'HOOD', 'UBER', 'LYFT', 'LUNR', 'XBIO', 'KDK']

# Results tracking
results = {
    'api': [],
    'ticker': [],
    'success': [],
    'latency_sec': [],
    'rows_returned': [],
    'data_start': [],
    'data_end': [],
    'gaps_detected': [],
    'error_msg': []
}

def log_result(api, ticker, success, latency, df=None, error=None):
    """Log test result"""
    results['api'].append(api)
    results['ticker'].append(ticker)
    results['success'].append(success)
    results['latency_sec'].append(round(latency, 3))
    
    if df is not None and len(df) > 0:
        results['rows_returned'].append(len(df))
        results['data_start'].append(df.index[0].strftime('%Y-%m-%d') if hasattr(df.index[0], 'strftime') else str(df.index[0]))
        results['data_end'].append(df.index[-1].strftime('%Y-%m-%d') if hasattr(df.index[-1], 'strftime') else str(df.index[-1]))
        # Check for gaps (more than 5 business days between rows)
        if len(df) > 1:
            gaps = (df.index[1:] - df.index[:-1]).days > 5
            results['gaps_detected'].append(gaps.sum() if hasattr(gaps, 'sum') else 0)
        else:
            results['gaps_detected'].append(0)
    else:
        results['rows_returned'].append(0)
        results['data_start'].append(None)
        results['data_end'].append(None)
        results['gaps_detected'].append(0)
    
    results['error_msg'].append(str(error) if error else None)

print("✓ Setup complete")
print(f"✓ Testing {len(WATCHLIST)} tickers: {', '.join(WATCHLIST)}")

## Test 1: yfinance (Baseline - Free & Unlimited)
This is our fallback. Must work 100% or we're dead in the water.

In [ ]:
# Test 1: yfinance
try:
    import yfinance as yf
    print("Testing yfinance...")
    
    for ticker in WATCHLIST:
        try:
            t0 = time.time()
            df = yf.download(ticker, period='3mo', interval='1d', progress=False)
            latency = time.time() - t0
            
            if len(df) > 0:
                log_result('yfinance', ticker, True, latency, df)
                print(f"  ✓ {ticker}: {len(df)} rows, {latency:.2f}s")
            else:
                log_result('yfinance', ticker, False, latency, error="No data returned")
                print(f"  ✗ {ticker}: No data")
        except Exception as e:
            log_result('yfinance', ticker, False, time.time() - t0, error=e)
            print(f"  ✗ {ticker}: {e}")
    
    # Summary
    yf_results = pd.DataFrame(results)
    yf_success_rate = yf_results[yf_results['api'] == 'yfinance']['success'].mean()
    yf_avg_latency = yf_results[yf_results['api'] == 'yfinance']['latency_sec'].mean()
    
    print(f"\nyfinance Summary:")
    print(f"  Success Rate: {yf_success_rate*100:.1f}%")
    print(f"  Avg Latency: {yf_avg_latency:.2f}s")
    print(f"  VERDICT: {'✓ PASS' if yf_success_rate >= 0.95 and yf_avg_latency < 5 else '✗ FAIL'}")
    
except ImportError:
    print("✗ yfinance not installed. Install: pip install yfinance")

## Test 2: Twelve Data (800 calls/day)
Premium free tier. Test if it's actually better than yfinance.

In [ ]:
# Test 2: Twelve Data
try:
    import requests
    
    TWELVE_DATA_KEY = API_KEYS.get('TWELVE_DATA_KEY')
    
    if TWELVE_DATA_KEY:
        print("Testing Twelve Data...")
        
        for ticker in WATCHLIST:
            try:
                t0 = time.time()
                url = f"https://api.twelvedata.com/time_series"
                params = {
                    'symbol': ticker,
                    'interval': '1day',
                    'outputsize': 90,
                    'apikey': TWELVE_DATA_KEY
                }
                response = requests.get(url, params=params)
                latency = time.time() - t0
                
                if response.status_code == 200:
                    data = response.json()
                    if 'values' in data:
                        df = pd.DataFrame(data['values'])
                        df['datetime'] = pd.to_datetime(df['datetime'])
                        df = df.set_index('datetime')
                        log_result('twelve_data', ticker, True, latency, df)
                        print(f"  ✓ {ticker}: {len(df)} rows, {latency:.2f}s")
                    else:
                        log_result('twelve_data', ticker, False, latency, error=data.get('message', 'No values'))
                        print(f"  ✗ {ticker}: {data.get('message', 'No values')}")
                else:
                    log_result('twelve_data', ticker, False, latency, error=f"HTTP {response.status_code}")
                    print(f"  ✗ {ticker}: HTTP {response.status_code}")
            except Exception as e:
                log_result('twelve_data', ticker, False, time.time() - t0, error=e)
                print(f"  ✗ {ticker}: {e}")
        
        # Summary
        df_results = pd.DataFrame(results)
        td_results = df_results[df_results['api'] == 'twelve_data']
        td_success_rate = td_results['success'].mean()
        td_avg_latency = td_results['latency_sec'].mean()
        
        print(f"\nTwelve Data Summary:")
        print(f"  Success Rate: {td_success_rate*100:.1f}%")
        print(f"  Avg Latency: {td_avg_latency:.2f}s")
        print(f"  VERDICT: {'✓ PASS' if td_success_rate >= 0.95 and td_avg_latency < 5 else '✗ FAIL'}")
    else:
        print("⊘ Twelve Data: No API key set. Skipping.")
        print("  Get free key: https://twelvedata.com/")
        
except ImportError:
    print("✗ requests not installed. Install: pip install requests")

## Test 3: Finnhub (60 calls/min)
Fast rate limit. Good for real-time. Test latency.

In [ ]:
# Test 3: Finnhub
try:
    import finnhub
    
    FINNHUB_KEY = API_KEYS.get('FINNHUB_KEY')
    
    if FINNHUB_KEY:
        print("Testing Finnhub...")
        client = finnhub.Client(api_key=FINNHUB_KEY)
        
        # Finnhub uses Unix timestamps
        end_date = int(datetime.now().timestamp())
        start_date = int((datetime.now() - timedelta(days=90)).timestamp())
        
        for ticker in WATCHLIST:
            try:
                t0 = time.time()
                candles = client.stock_candles(ticker, 'D', start_date, end_date)
                latency = time.time() - t0
                
                if candles and candles.get('s') == 'ok':
                    df = pd.DataFrame({
                        'timestamp': candles['t'],
                        'close': candles['c'],
                        'high': candles['h'],
                        'low': candles['l'],
                        'open': candles['o'],
                        'volume': candles['v']
                    })
                    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
                    df = df.set_index('timestamp')
                    log_result('finnhub', ticker, True, latency, df)
                    print(f"  ✓ {ticker}: {len(df)} rows, {latency:.2f}s")
                else:
                    log_result('finnhub', ticker, False, latency, error="No data or status != ok")
                    print(f"  ✗ {ticker}: No data")
            except Exception as e:
                log_result('finnhub', ticker, False, time.time() - t0, error=e)
                print(f"  ✗ {ticker}: {e}")
            
            # Rate limit: 60/min = 1 per second, add small delay
            time.sleep(1.1)
        
        # Summary
        df_results = pd.DataFrame(results)
        fh_results = df_results[df_results['api'] == 'finnhub']
        fh_success_rate = fh_results['success'].mean()
        fh_avg_latency = fh_results['latency_sec'].mean()
        
        print(f"\nFinnhub Summary:")
        print(f"  Success Rate: {fh_success_rate*100:.1f}%")
        print(f"  Avg Latency: {fh_avg_latency:.2f}s")
        print(f"  VERDICT: {'✓ PASS' if fh_success_rate >= 0.95 and fh_avg_latency < 5 else '✗ FAIL'}")
    else:
        print("⊘ Finnhub: No API key set. Skipping.")
        print("  Get free key: https://finnhub.io/")
        
except ImportError:
    print("✗ finnhub-python not installed. Install: pip install finnhub-python")

## Test 4: Alpha Vantage (500 calls/day)
Older API, free tier is limited. Test if worth keeping.

In [ ]:
# Test 7: EODHD
try:
    import requests
    
    EODHD_KEY = API_KEYS.get('EODHD_KEY')
    
    if EODHD_KEY:
        print("Testing EODHD...")
        
        # EODHD uses .US suffix for US stocks
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')
        
        for ticker in WATCHLIST:
            try:
                t0 = time.time()
                url = f"https://eodhd.com/api/eod/{ticker}.US"
                params = {
                    'api_token': EODHD_KEY,
                    'from': start_date,
                    'to': end_date,
                    'fmt': 'json'
                }
                response = requests.get(url, params=params)
                latency = time.time() - t0
                
                if response.status_code == 200:
                    data = response.json()
                    if data and len(data) > 0:
                        df = pd.DataFrame(data)
                        df['date'] = pd.to_datetime(df['date'])
                        df = df.set_index('date')
                        log_result('eodhd', ticker, True, latency, df)
                        print(f"  ✓ {ticker}: {len(df)} rows, {latency:.2f}s")
                    else:
                        log_result('eodhd', ticker, False, latency, error="No data returned")
                        print(f"  ✗ {ticker}: No data returned")
                else:
                    log_result('eodhd', ticker, False, latency, error=f"HTTP {response.status_code}")
                    print(f"  ✗ {ticker}: HTTP {response.status_code}")
            except Exception as e:
                log_result('eodhd', ticker, False, time.time() - t0, error=e)
                print(f"  ✗ {ticker}: {e}")
        
        # Summary
        df_results = pd.DataFrame(results)
        eodhd_results = df_results[df_results['api'] == 'eodhd']
        eodhd_success_rate = eodhd_results['success'].mean()
        eodhd_avg_latency = eodhd_results['latency_sec'].mean()
        
        print(f"\nEODHD Summary:")
        print(f"  Success Rate: {eodhd_success_rate*100:.1f}%")
        print(f"  Avg Latency: {eodhd_avg_latency:.2f}s")
        print(f"  VERDICT: {'✓ PASS' if eodhd_success_rate >= 0.95 and eodhd_avg_latency < 5 else '✗ FAIL'}")
    else:
        print("⊘ EODHD: No API key found in .env")
        
except ImportError:
    print("✗ requests not installed. Install: pip install requests")

## Test 7: EODHD (20 calls/day)
Most limited free tier. Test if worth the quota.

In [ ]:
# Test 6: FMP (Financial Modeling Prep)
try:
    import requests
    
    FMP_KEY = API_KEYS.get('FMP_KEY')
    
    if FMP_KEY:
        print("Testing FMP...")
        
        for ticker in WATCHLIST:
            try:
                t0 = time.time()
                url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{ticker}"
                params = {'apikey': FMP_KEY}
                response = requests.get(url, params=params)
                latency = time.time() - t0
                
                if response.status_code == 200:
                    data = response.json()
                    if 'historical' in data and data['historical']:
                        df = pd.DataFrame(data['historical'])
                        df['date'] = pd.to_datetime(df['date'])
                        df = df.set_index('date')
                        df = df.sort_index()
                        # Get last 90 days
                        df = df.tail(90)
                        log_result('fmp', ticker, True, latency, df)
                        print(f"  ✓ {ticker}: {len(df)} rows, {latency:.2f}s")
                    else:
                        log_result('fmp', ticker, False, latency, error="No historical data")
                        print(f"  ✗ {ticker}: No historical data")
                else:
                    log_result('fmp', ticker, False, latency, error=f"HTTP {response.status_code}")
                    print(f"  ✗ {ticker}: HTTP {response.status_code}")
            except Exception as e:
                log_result('fmp', ticker, False, time.time() - t0, error=e)
                print(f"  ✗ {ticker}: {e}")
        
        # Summary
        df_results = pd.DataFrame(results)
        fmp_results = df_results[df_results['api'] == 'fmp']
        fmp_success_rate = fmp_results['success'].mean()
        fmp_avg_latency = fmp_results['latency_sec'].mean()
        
        print(f"\nFMP Summary:")
        print(f"  Success Rate: {fmp_success_rate*100:.1f}%")
        print(f"  Avg Latency: {fmp_avg_latency:.2f}s")
        print(f"  VERDICT: {'✓ PASS' if fmp_success_rate >= 0.95 and fmp_avg_latency < 5 else '✗ FAIL'}")
    else:
        print("⊘ FMP: No API key found in .env")
        
except ImportError:
    print("✗ requests not installed. Install: pip install requests")

## Test 6: FMP (250 calls/day)
Financial Modeling Prep. Good fundamentals data.

In [ ]:
# Test 5: Polygon.io
try:
    import requests
    
    POLYGON_KEY = API_KEYS.get('POLYGON_KEY')
    
    if POLYGON_KEY:
        print("Testing Polygon.io...")
        
        # Polygon uses YYYY-MM-DD format
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')
        
        for ticker in WATCHLIST:
            try:
                t0 = time.time()
                url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/day/{start_date}/{end_date}"
                params = {'apiKey': POLYGON_KEY}
                response = requests.get(url, params=params)
                latency = time.time() - t0
                
                if response.status_code == 200:
                    data = response.json()
                    if 'results' in data and data['results']:
                        df = pd.DataFrame(data['results'])
                        df['timestamp'] = pd.to_datetime(df['t'], unit='ms')
                        df = df.set_index('timestamp')
                        df = df.rename(columns={'o': 'open', 'h': 'high', 'l': 'low', 'c': 'close', 'v': 'volume'})
                        log_result('polygon', ticker, True, latency, df)
                        print(f"  ✓ {ticker}: {len(df)} rows, {latency:.2f}s")
                    else:
                        log_result('polygon', ticker, False, latency, error=data.get('status', 'No results'))
                        print(f"  ✗ {ticker}: {data.get('status', 'No results')}")
                else:
                    log_result('polygon', ticker, False, latency, error=f"HTTP {response.status_code}")
                    print(f"  ✗ {ticker}: HTTP {response.status_code}")
            except Exception as e:
                log_result('polygon', ticker, False, time.time() - t0, error=e)
                print(f"  ✗ {ticker}: {e}")
            
            # Rate limit: 5 calls/min, add delay
            time.sleep(13)
        
        # Summary
        df_results = pd.DataFrame(results)
        poly_results = df_results[df_results['api'] == 'polygon']
        poly_success_rate = poly_results['success'].mean()
        poly_avg_latency = poly_results['latency_sec'].mean()
        
        print(f"\nPolygon.io Summary:")
        print(f"  Success Rate: {poly_success_rate*100:.1f}%")
        print(f"  Avg Latency: {poly_avg_latency:.2f}s")
        print(f"  VERDICT: {'✓ PASS' if poly_success_rate >= 0.95 and poly_avg_latency < 5 else '✗ FAIL'}")
    else:
        print("⊘ Polygon.io: No API key found in .env")
        
except ImportError:
    print("✗ requests not installed. Install: pip install requests")

## Test 5: Polygon.io (Free Tier)
Delayed data, 5 calls/min. Test if useful for historical.

In [ ]:
# Test 4: Alpha Vantage
try:
    import requests
    
    ALPHA_VANTAGE_KEY = API_KEYS.get('ALPHA_VANTAGE_KEY')
    
    if ALPHA_VANTAGE_KEY:
        print("Testing Alpha Vantage...")
        
        for ticker in WATCHLIST:
            try:
                t0 = time.time()
                url = 'https://www.alphavantage.co/query'
                params = {
                    'function': 'TIME_SERIES_DAILY',
                    'symbol': ticker,
                    'outputsize': 'compact',  # 100 days
                    'apikey': ALPHA_VANTAGE_KEY
                }
                response = requests.get(url, params=params)
                latency = time.time() - t0
                
                if response.status_code == 200:
                    data = response.json()
                    if 'Time Series (Daily)' in data:
                        ts = data['Time Series (Daily)']
                        df = pd.DataFrame.from_dict(ts, orient='index')
                        df.index = pd.to_datetime(df.index)
                        df = df.sort_index()
                        df.columns = ['open', 'high', 'low', 'close', 'volume']
                        df = df.astype(float)
                        log_result('alpha_vantage', ticker, True, latency, df)
                        print(f"  ✓ {ticker}: {len(df)} rows, {latency:.2f}s")
                    else:
                        error_msg = data.get('Note', data.get('Error Message', 'Unknown error'))
                        log_result('alpha_vantage', ticker, False, latency, error=error_msg)
                        print(f"  ✗ {ticker}: {error_msg}")
                else:
                    log_result('alpha_vantage', ticker, False, latency, error=f"HTTP {response.status_code}")
                    print(f"  ✗ {ticker}: HTTP {response.status_code}")
            except Exception as e:
                log_result('alpha_vantage', ticker, False, time.time() - t0, error=e)
                print(f"  ✗ {ticker}: {e}")
            
            # Rate limit: 5 calls/min free tier, add delay
            time.sleep(13)
        
        # Summary
        df_results = pd.DataFrame(results)
        av_results = df_results[df_results['api'] == 'alpha_vantage']
        av_success_rate = av_results['success'].mean()
        av_avg_latency = av_results['latency_sec'].mean()
        
        print(f"\nAlpha Vantage Summary:")
        print(f"  Success Rate: {av_success_rate*100:.1f}%")
        print(f"  Avg Latency: {av_avg_latency:.2f}s")
        print(f"  VERDICT: {'✓ PASS' if av_success_rate >= 0.95 and av_avg_latency < 5 else '✗ FAIL'}")
    else:
        print("⊘ Alpha Vantage: No API key set. Skipping.")
        print("  Get free key: https://www.alphavantage.co/support/#api-key")
        
except ImportError:
    print("✗ requests not installed. Install: pip install requests")

## Final Results: Which APIs Actually Work?
Compare all APIs head-to-head. Keep only what passes acceptance criteria.

In [ ]:
# Comprehensive Results
df_results = pd.DataFrame(results)

print("\n" + "="*80)
print("FINAL API VALIDATION RESULTS")
print("="*80)

# Summary by API
summary = df_results.groupby('api').agg({
    'success': ['sum', 'count', 'mean'],
    'latency_sec': ['mean', 'max'],
    'rows_returned': 'mean',
    'gaps_detected': 'sum'
}).round(3)

summary.columns = ['Successes', 'Total Tests', 'Success Rate', 'Avg Latency (s)', 'Max Latency (s)', 'Avg Rows', 'Total Gaps']

print("\nAPI Performance Summary:")
print(summary)

# Pass/Fail Verdict
print("\n" + "-"*80)
print("ACCEPTANCE CRITERIA: Success Rate ≥95%, Avg Latency <5s")
print("-"*80)

for api in df_results['api'].unique():
    api_data = df_results[df_results['api'] == api]
    success_rate = api_data['success'].mean()
    avg_latency = api_data['latency_sec'].mean()
    
    passes = success_rate >= 0.95 and avg_latency < 5
    verdict = "✓ PASS - USE IN PRODUCTION" if passes else "✗ FAIL - REJECT"
    
    print(f"\n{api.upper():<20} {verdict}")
    print(f"  Success Rate: {success_rate*100:.1f}% {'✓' if success_rate >= 0.95 else '✗'}")
    print(f"  Avg Latency:  {avg_latency:.2f}s {'✓' if avg_latency < 5 else '✗'}")
    
    if api_data['gaps_detected'].sum() > 0:
        print(f"  ⚠️  Data Gaps:   {api_data['gaps_detected'].sum()} gaps detected")

# Detailed failures
failures = df_results[df_results['success'] == False]
if len(failures) > 0:
    print("\n" + "-"*80)
    print("DETAILED FAILURES:")
    print("-"*80)
    for _, row in failures.iterrows():
        print(f"\n{row['api'].upper()} - {row['ticker']}")
        print(f"  Error: {row['error_msg']}")
        print(f"  Latency: {row['latency_sec']:.2f}s")

## Save Results for Next Notebook
Export test results to use in subsequent validation notebooks.

In [ ]:
# Save results
df_results.to_csv('../data/01_api_validation_results.csv', index=False)
print("\n✓ Results saved to: data/01_api_validation_results.csv")

# Create approved APIs list
approved_apis = []
for api in df_results['api'].unique():
    api_data = df_results[df_results['api'] == api]
    success_rate = api_data['success'].mean()
    avg_latency = api_data['latency_sec'].mean()
    
    if success_rate >= 0.95 and avg_latency < 5:
        approved_apis.append(api)

print(f"\n✓ Approved APIs for production: {approved_apis}")

# Save approved list
with open('../data/approved_apis.txt', 'w') as f:
    f.write('\n'.join(approved_apis))

print("\n" + "="*80)
print("NEXT STEPS:")
print("="*80)
print("1. Set API keys for any APIs you want to test (skipped APIs above)")
print("2. Re-run this notebook to validate those APIs")
print("3. Move to notebook 02: Dark Pool Signal Validation")
print("4. Use ONLY approved APIs from this test in future notebooks")

## API Key Setup (For Reference)
Run these in terminal or set in your environment:

```bash
# yfinance - no key needed

# Twelve Data (800/day free)
export TWELVE_DATA_KEY="your_key_here"

# Finnhub (60/min free)
export FINNHUB_KEY="your_key_here"

# Alpha Vantage (500/day free)
export ALPHA_VANTAGE_KEY="your_key_here"

# Or set in Python:
# import os
# os.environ['TWELVE_DATA_KEY'] = 'your_key_here'
```

Get free keys:
- Twelve Data: https://twelvedata.com/
- Finnhub: https://finnhub.io/
- Alpha Vantage: https://www.alphavantage.co/support/#api-key